# Retail Media Intelligence: Exploratory Analysis

This notebook queries the PostgreSQL warehouse through SQLAlchemy and uses the same reusable Plotly functions as the standalone dashboard generator. It does not read raw CSV files.

Prerequisites:

1. Load the generated data into PostgreSQL with `python src/etl/load_to_postgres.py --create-schema`.
2. Create `.env` from `.env.example` and set valid PostgreSQL credentials.
3. Install the project requirements.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.plotly_dashboard import (
    build_all_figures,
    create_postgres_engine,
    export_figures,
    load_dashboard_data,
)

OUTPUT_DIR = PROJECT_ROOT / "dashboards" / "plotly"
engine = create_postgres_engine(PROJECT_ROOT / ".env")

## Load analysis datasets

The query layer aggregates the 9M-row event fact inside PostgreSQL. The attribution comparison also pulls journeys and channel spend from PostgreSQL before running Last Touch, Linear, Markov, and Shapley models.

In [ ]:
data = load_dashboard_data(engine)
figures = build_all_figures(data)

{
    "funnel_rows": len(data.funnel),
    "daily_rows": len(data.daily_economics),
    "attribution_rows": len(data.attribution),
    "cohort_rows": len(data.cohorts),
    "rfm_customers": len(data.rfm),
    "budget_days": len(data.budget_pacing),
}

## 1. Funnel conversion by campaign type

The grouped funnel makes the scale difference between awareness and conversion explicit while preserving campaign-type comparisons. Hover over a stage to see its percentage of initial impressions.

In [ ]:
figures["01_campaign_funnel.html"].show()

## 2. Daily spend, revenue, and promotional seasonality

Spend and revenue use separate axes because their units differ materially. The shaded windows identify the back-to-school and holiday demand shocks embedded in the synthetic data.

In [ ]:
figures["02_daily_spend_vs_revenue.html"].show()

## 3. Attribution model comparison

This is the central model-risk view. Last Touch shows where demand was captured; Linear spreads credit evenly; Markov measures transition removal effects; Shapley measures average marginal coalition value. Large differences are a reason to test assumptions, not to hide model disagreement.

In [ ]:
figures["03_attribution_model_comparison.html"].show()

## 4. Cohort retention heatmap

Rows are signup cohorts and columns are months after signup. Restricting the SQL to fully matured cohorts prevents recent customers from being treated as churned before they have had six months of observation.

In [ ]:
figures["04_cohort_retention_heatmap.html"].show()

## 5. RFM customer segmentation

Recency and frequency define the axes, monetary value controls marker size, and the SQL-assigned RFM segment controls color. The chart uses WebGL so tens of thousands of customers remain interactive.

In [ ]:
figures["05_rfm_segment_scatter.html"].show()

## 6. Budget pacing control chart

The default query chooses the campaign with the most severe latest pacing deviation. The solid line is actual spend, the dashed line is budget, the smoothed line is the rolling seven-day average, and red diamonds mark warning or critical days.

In [ ]:
figures["06_budget_pacing_control_chart.html"].show()

## Export standalone HTML files

Each export embeds Plotly.js, so the file can be opened directly without rerunning this notebook or requiring an internet connection.

In [ ]:
exported_paths = export_figures(figures, OUTPUT_DIR)
exported_paths

In [ ]:
engine.dispose()